In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import torch

from transformers import T5Tokenizer, T5ForConditionalGeneration

CSV_PATH = "/content/drive/MyDrive/spells_master.csv"
MODEL_NAME = "t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [4]:
df = pd.read_csv(CSV_PATH)

print(df.shape)
df.head(3)

(1333, 20)


,name,desc,higher_levels,level,school,classes,subclasses,cast_time,range,duration,verbal,somatic,material,material_desc,concentration,ritual,damage_type,dc_type,attack_type,source
0,Prismatic Wall,"A shimmering, multicolored plane of light form...",NaN,9,Abjuration,['Wizard'],[],1 action,60 feet,10 minutes,True,True,False,NaN,False,False,NaN,NaN,NaN,wotc-srd
1,Symbol,"When you cast this spell, you inscribe a harmf...",NaN,7,Abjuration,"['Bard', 'Cleric', 'Wizard']",[],1 minute,Touch,Until dispelled or triggered,True,True,True,"Mercury, phosphorus, and powdered diamond and ...",False,False,NaN,NaN,NaN,wotc-srd
2,Teleport,This spell instantly transports you and up to ...,NaN,7,Conjuration,"['Bard', 'Sorcerer', 'Wizard']",[],1 action,10 feet,Instantaneous,True,False,False,NaN,False,False,NaN,NaN,NaN,wotc-srd


In [5]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(device)

print("model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model loaded


In [6]:
prompt = "describe spell: Fireball"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40
    )

result = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(result)

spell: Fireball. Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball


In [7]:
df = df.dropna(subset=["name", "desc"])

df["name"] = df["name"].astype(str)
df["desc"] = df["desc"].astype(str)

df = df[df["desc"].str.len() > 20]

print(len(df))

1333


In [8]:
train_pairs = []

for _, row in df.iterrows():
    prompt = f"describe spell: {row['name']}"
    target = row["desc"]

    train_pairs.append({
        "input": prompt,
        "target": target
    })

train_pairs[:2]

[{'input': 'describe spell: Prismatic Wall',
  'target': "A shimmering, multicolored plane of light forms a vertical opaque wall-up to 90 feet long, 30 feet high, and 1 inch thick-centered on a point you can see within range. Alternatively, you can shape the wall into a sphere up to 30 feet in diameter centered on a point you choose within range. The wall remains in place for the duration. If you position the wall so that it passes through a space occupied by a creature, the spell fails, and your action and the spell slot are wasted. The wall sheds bright light out to a range of 100 feet and dim light for an additional 100 feet. You and creatures you designate at the time you cast the spell can pass through and remain near the wall without harm. If another creature that can see the wall moves to within 20 feet of it or starts its turn there, the creature must succeed on a constitution saving throw or become blinded for 1 minute. The wall consists of seven layers, each with a different 

In [9]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_pairs,
    test_size=0.1,
    random_state=49
)

print(len(train_data))
print(len(val_data))

1199
134


In [25]:
class SpellDataset(torch.utils.data.Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        model_inputs = tokenizer(
            item["input"],
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        labels = tokenizer(
            item["target"],
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == tokenizer.pad_token_id] = -100  # ignore padding in loss

        return {
            "input_ids": model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels": label_ids
        }

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8
)

print(len(train_loader))

150


In [12]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4
)

model.train()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

loss = outputs.loss

print(loss.item())

13.277107238769531


In [24]:
EPOCHS = 15

train_losses = []
val_losses = []

In [15]:
for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

Epoch 1 | Train Loss: 2.3220 | Val Loss: 1.7539
Epoch 2 | Train Loss: 1.6501 | Val Loss: 1.5985
Epoch 3 | Train Loss: 1.4847 | Val Loss: 1.5243
Epoch 4 | Train Loss: 1.3784 | Val Loss: 1.4785
Epoch 5 | Train Loss: 1.2877 | Val Loss: 1.4556


In [15]:
from transformers import get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler

SAVE_DIR = "/content/drive/MyDrive/GenAiText/t5_spell_model"

os.makedirs(SAVE_DIR, exist_ok=True)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

scaler = GradScaler()

best_val_loss = float("inf")
start_epoch = 0

/tmp/ipykernel_17247/284320953.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [16]:
checkpoint_path = os.path.join(
    SAVE_DIR,
    "training_state.pt"
)

if os.path.exists(checkpoint_path):

    print("resuming previous training run")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )

    scaler.load_state_dict(
        checkpoint["scaler_state"]
    )

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]

    print(f"starting from epoch {start_epoch}")

else:

    print("starting fresh training run")

starting fresh training run


In [17]:
for epoch in range(start_epoch, EPOCHS):

    model.train()

    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with autocast():

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with autocast():

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print(
        f"Train Loss: {avg_train_loss:.4f}"
    )

    print(
        f"Val Loss: {avg_val_loss:.4f}"
    )

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_val_loss": best_val_loss
            },
            checkpoint_path
        )

        print("saved new best model")

    print("-" * 40)

/tmp/ipykernel_17247/3190387456.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_17247/3190387456.py:30: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
/tmp/ipykernel_17247/3190387456.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5
Train Loss: 3.7181
Val Loss: 1.8830


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved new best model
----------------------------------------
Epoch 2/5
Train Loss: 1.7903
Val Loss: 1.6968


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

saved new best model
----------------------------------------
Epoch 4/5
Train Loss: 1.5479
Val Loss: 1.5899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved new best model
----------------------------------------
Epoch 5/5
Train Loss: 1.5055
Val Loss: 1.5801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved new best model
----------------------------------------


In [18]:
def build_spell_prompt(row):

    fields = []

    fields.append(f"level: {row['level']}")

    if "school" in row and pd.notna(row["school"]):
        fields.append(f"school: {row['school']}")

    if "cast_time" in row and pd.notna(row["cast_time"]):
        fields.append(f"cast_time: {row['cast_time']}")

    if "range" in row and pd.notna(row["range"]):
        fields.append(f"range: {row['range']}")

    if "duration" in row and pd.notna(row["duration"]):
        fields.append(f"duration: {row['duration']}")

    if "damage_type" in row and pd.notna(row["damage_type"]):
        fields.append(f"damage: {row['damage_type']}")

    return " | ".join(fields)

In [19]:
multi_task_examples = []

for _, row in df.iterrows():

    name = str(row["name"])
    desc = str(row["desc"])

    attrs = build_spell_prompt(row)

    multi_task_examples.append({
        "input": f"describe spell: {name}",
        "target": desc,
        "task": "name_to_desc"
    })

    multi_task_examples.append({
        "input": f"generate name: {desc}",
        "target": name,
        "task": "desc_to_name"
    })

    multi_task_examples.append({
        "input": f"generate description: {attrs}",
        "target": desc,
        "task": "attr_to_desc"
    })

print(len(multi_task_examples))

pd.DataFrame(multi_task_examples).head()

3999


,input,target,task
0,describe spell: Prismatic Wall,"A shimmering, multicolored plane of light form...",name_to_desc
1,"generate name: A shimmering, multicolored plan...",Prismatic Wall,desc_to_name
2,generate description: level: 9 | school: Abjur...,"A shimmering, multicolored plane of light form...",attr_to_desc
3,describe spell: Symbol,"When you cast this spell, you inscribe a harmf...",name_to_desc
4,"generate name: When you cast this spell, you i...",Symbol,desc_to_name


In [20]:
multi_train, multi_val = train_test_split(
    multi_task_examples,
    test_size=0.1,
    random_state=42
)

train_dataset = SpellDataset(multi_train)
val_dataset = SpellDataset(multi_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

print(len(train_loader))
print(len(val_loader))

450
50


In [21]:
import math

train_history = []
val_history = []
perplexity_history = []

In [ ]:
# reinitialize everything clean before training
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler('cuda')

start_epoch = 0
best_val_loss = float("inf")
train_history = []
val_history = []
perplexity_history = []

print("ready")

In [26]:
train_history = []
val_history = []
perplexity_history = []
best_val_loss = float("inf")

for epoch in range(start_epoch, EPOCHS):

    model.train()
    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_history.append(avg_train_loss)

    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_history.append(avg_val_loss)

    perplexity = math.exp(avg_val_loss)
    perplexity_history.append(perplexity)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                "best_val_loss": best_val_loss,
                "train_history": train_history,
                "val_history": val_history,
                "perplexity_history": perplexity_history
            },
            checkpoint_path
        )
        print("saved best model")



Epoch 3/15
Train Loss: nan
Val Loss:   2.9879
Perplexity: 19.84


KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(perplexity_history)

plt.xlabel("Epoch")
plt.ylabel("Perplexity")

plt.title("Validation Perplexity")

plt.show()

In [ ]:
def generate(prompt, max_new_tokens=120):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [ ]:
probe_prompts = [

    "describe spell: Fireball",

    "describe spell: Misty Step",

    (
        "generate description: "
        "level: 3 | school: Necromancy | "
        "damage: Poison | duration: 1 minute"
    ),

    (
        "generate description: "
        "level: 7 | school: Conjuration | "
        "range: 500 feet | duration: Instantaneous"
    ),

    (
        "generate description: "
        "level: 2 | school: Illusion | "
        "duration: 10 minutes"
    ),

    (
        "generate description: "
        "level: 0 | school: Evocation | "
        "damage: Lightning | range: 60 feet"
    ),

    (
        "generate description: "
        "level: 6 | school: Transmutation | "
        "duration: 24 hours"
    ),

    (
        "generate name: "
        "A wave of freezing wind blasts outward from you, "
        "dealing cold damage to creatures in a cone."
    ),

    (
        "generate name: "
        "You summon spectral chains that restrain enemies "
        "and drain their life force."
    ),

    (
        "describe spell: Ashen Nova | "
        "school: Evocation | level: 5"
    ),

]

In [ ]:
for i, prompt in enumerate(probe_prompts):


    print(f"Prompt {i + 1}")

    print(prompt)

    print("Output:\n")

    result = generate(
        prompt,
        max_new_tokens=200
    )

    print(result)

    print("\n")